# The Hippies and Punks Problem

## Introduction

The *Hippies and Punks* problem is a variation of the classic *Missionaries and Cannibals* problem.  
The challenge is to transport a group of hippies and punks from one side of the river to the other, ensuring that **the punks are never in the majority relative to the hippies** on either side of the river. Otherwise, the punks rebel and overpower the hippies!

## Problem Definition

- There are **3 hippies** and **3 punks** on the left side of the river.
- A **boat** can carry at most **2 people** at a time and requires at least **1 operator**.
- The goal is to transport all the hippies and punks to the other side of the river without the punks ever being in the majority at any point.
- If, on either side of the river, there are more punks than hippies, the hippies will be "overpowered," and the attempt fails.

## Algorithm to Solve the Problem

1. **Define the possible states**: We represent the different valid configurations of people on both sides of the river.
2. **Define the possible moves**: Since the boat can carry up to 2 people, we list all possible transportation combinations.
3. **Generate valid successors**: For each valid state, we generate the next states while ensuring that the punks are never in the majority.
4. **Solve the problem**: We can use depth-first search (DFS) or breadth-first search (BFS) to find a solution.

Let us begin by implementing the solution step by step.

## Pseudocode

Here is a sketch of the algorithm to solve the problem:

```
1. Define the initial state: (3 hippies, 3 punks, boat on the left side)
2. Create a list of valid boat moves
3. While there are states to explore:
   a. Take the next state
   b. For each possible move:
      i. Compute the resulting new state
      ii. Check whether the state is valid (punks can never be the majority)
      iii. If the state is the goal, terminate the algorithm
      iv. Otherwise, add it to the list of states to explore
4. If we exhaust all states without finding a solution, the problem is unsolvable.
```

In [1]:
import copy

# Geração de estados válidos
valid_states = []
for punks_left in range(4):
    for hippies_left in range(4):
        punks_right = 3 - punks_left
        hippies_right = 3 - hippies_left
        for boat in ['left', 'right']:
            # Um estado é válido quando os hippies nunca ficam em minoria
            if ((hippies_left >= punks_left or hippies_left == 0) and
                (hippies_right >= punks_right or hippies_right == 0)):
                state = {
                    'punks_left': punks_left,
                    'hippies_left': hippies_left,
                    'punks_right': punks_right,
                    'hippies_right': hippies_right,
                    'boat': boat
                }
                valid_states.append(state)

# Exibir alguns estados válidos
valid_states[:5]  # Mostrando os primeiros 5 estados gerados


[{'punks_left': 0,
  'hippies_left': 0,
  'punks_right': 3,
  'hippies_right': 3,
  'boat': 'left'},
 {'punks_left': 0,
  'hippies_left': 0,
  'punks_right': 3,
  'hippies_right': 3,
  'boat': 'right'},
 {'punks_left': 0,
  'hippies_left': 3,
  'punks_right': 3,
  'hippies_right': 0,
  'boat': 'left'},
 {'punks_left': 0,
  'hippies_left': 3,
  'punks_right': 3,
  'hippies_right': 0,
  'boat': 'right'},
 {'punks_left': 1,
  'hippies_left': 0,
  'punks_right': 2,
  'hippies_right': 3,
  'boat': 'left'}]

## Possible Moves

The boat can carry at most **2 people** at a time. The allowed passenger combinations are:

1. (1,0) → Take 1 punk
2. (0,1) → Take 1 hippie
3. (1,1) → Take 1 punk and 1 hippie
4. (2,0) → Take 2 punks
5. (0,2) → Take 2 hippies

Let us define these moves as a list of tuples:

In [2]:
# Definição dos movimentos possíveis
# Cada tupla representa (punks_no_barco, hippies_no_barco)
moves = [(1,0), (0,1), (1,1), (2,0), (0,2)]
moves


[(1, 0), (0, 1), (1, 1), (2, 0), (0, 2)]

## Function to Generate Successor States

Now, we create a function to generate the valid successor states from a current state, ensuring that **the punks are never in the majority on either side of the river**.

In [3]:
def get_valid_successor_states_and_moves(current_state, moves):
    """Return all valid successor states together with the move used."""
    valid_successors = []

    for punks_move, hippies_move in moves:
        # copiamos o estado atual para não modificar o original
        new_state = copy.deepcopy(current_state)

        if current_state['boat'] == 'left':
            # barco sai da esquerda para a direita
            new_state['punks_left'] -= punks_move
            new_state['hippies_left'] -= hippies_move
            new_state['punks_right'] += punks_move
            new_state['hippies_right'] += hippies_move
            new_state['boat'] = 'right'
        else:
            # barco volta da direita para a esquerda
            new_state['punks_left'] += punks_move
            new_state['hippies_left'] += hippies_move
            new_state['punks_right'] -= punks_move
            new_state['hippies_right'] -= hippies_move
            new_state['boat'] = 'left'

        # checando limites válidos de pessoas em cada margem
        counts_ok = all(0 <= new_state[key] <= 3 for key in [
            'punks_left', 'hippies_left', 'punks_right', 'hippies_right'
        ])

        if not counts_ok:
            continue

        # checando a restrição principal do problema
        left_ok = (new_state['hippies_left'] >= new_state['punks_left'] or new_state['hippies_left'] == 0)
        right_ok = (new_state['hippies_right'] >= new_state['punks_right'] or new_state['hippies_right'] == 0)

        if left_ok and right_ok:
            valid_successors.append((new_state, (punks_move, hippies_move)))

    return valid_successors


## Testing the Generation of Valid States

Let us test the function with an initial state where all the hippies and punks are on the left side of the river.

In [4]:
initial_state = {
    'punks_left': 3,
    'hippies_left': 3,
    'punks_right': 0,
    'hippies_right': 0,
    'boat': 'left'
}

successors = get_valid_successor_states_and_moves(initial_state, moves)
successors


[({'punks_left': 2,
   'hippies_left': 3,
   'punks_right': 1,
   'hippies_right': 0,
   'boat': 'right'},
  (1, 0)),
 ({'punks_left': 2,
   'hippies_left': 2,
   'punks_right': 1,
   'hippies_right': 1,
   'boat': 'right'},
  (1, 1)),
 ({'punks_left': 1,
   'hippies_left': 3,
   'punks_right': 2,
   'hippies_right': 0,
   'boat': 'right'},
  (2, 0))]

## BFS

- Create a QUEUE to store the states to be explored, starting with the initial state.
- Create a SET of visited states to avoid repetitions.
- While there are states in the QUEUE:
   - Remove the first state from the QUEUE.
   - If the state is the GOAL STATE (all hippies and punks on the right bank), RETURN the path to it.
   
   - For each allowed move:
      - Generate a NEW STATE by applying the move.
      - If the NEW STATE is valid and has not yet been visited:
         - Add the NEW STATE to the QUEUE.
         - Mark the state as VISITED.
- If the QUEUE becomes empty without finding a solution, RETURN "No solution found".

In [5]:
# Estado inicial que será usado pelo BFS
initial_state


{'punks_left': 3,
 'hippies_left': 3,
 'punks_right': 0,
 'hippies_right': 0,
 'boat': 'left'}

In [6]:
from collections import deque

def bfs(initial_state):
    # Fila de estados a explorar [(estado, caminho)]
    queue = deque([(initial_state, [])])

    # Conjunto de estados já visitados
    visited = set()

    while queue:
        # Explorar o estado mais próximo
        current_state, path = queue.popleft()

        # Transformamos o dicionário em tupla para poder usar no set
        state_key = (
            current_state['punks_left'],
            current_state['hippies_left'],
            current_state['punks_right'],
            current_state['hippies_right'],
            current_state['boat']
        )

        # Se o estado já foi visitado, ignoramos
        if state_key in visited:
            continue
        visited.add(state_key)

        # Se chegamos ao estado final, retornamos o caminho até ele
        if (current_state['punks_left'] == 0 and
            current_state['hippies_left'] == 0 and
            current_state['punks_right'] == 3 and
            current_state['hippies_right'] == 3):
            return path

        # Gerar sucessores válidos
        for successor, move in get_valid_successor_states_and_moves(current_state, moves):
            successor_key = (
                successor['punks_left'],
                successor['hippies_left'],
                successor['punks_right'],
                successor['hippies_right'],
                successor['boat']
            )

            # Adicionar sucessor à fila apenas se ainda não foi visitado
            if successor_key not in visited:
                queue.append((successor, path + [move]))

    # Se não encontrar solução
    return None

# Estado inicial
initial_state = {
    'punks_left': 3,
    'hippies_left': 3,
    'punks_right': 0,
    'hippies_right': 0,
    'boat': 'left'
}

solution_path = bfs(initial_state)

print('Sequência de movimentos para solução ótima:', solution_path)


Sequência de movimentos para solução ótima: [(1, 1), (0, 1), (2, 0), (1, 0), (0, 2), (1, 1), (0, 2), (1, 0), (2, 0), (1, 0), (2, 0)]


In [7]:
def apply_move(state, move):
    """Apply one move and return the next state."""
    punks_move, hippies_move = move
    new_state = copy.deepcopy(state)

    if state['boat'] == 'left':
        new_state['punks_left'] -= punks_move
        new_state['hippies_left'] -= hippies_move
        new_state['punks_right'] += punks_move
        new_state['hippies_right'] += hippies_move
        new_state['boat'] = 'right'
    else:
        new_state['punks_left'] += punks_move
        new_state['hippies_left'] += hippies_move
        new_state['punks_right'] -= punks_move
        new_state['hippies_right'] -= hippies_move
        new_state['boat'] = 'left'

    return new_state

print('Passo a passo da solução:')
state = copy.deepcopy(initial_state)
print('Inicial:', state)

for step, move in enumerate(solution_path, start=1):
    state = apply_move(state, move)
    print(f'Passo {step} | move={move} -> {state}')


Passo a passo da solução:
Inicial: {'punks_left': 3, 'hippies_left': 3, 'punks_right': 0, 'hippies_right': 0, 'boat': 'left'}
Passo 1 | move=(1, 1) -> {'punks_left': 2, 'hippies_left': 2, 'punks_right': 1, 'hippies_right': 1, 'boat': 'right'}
Passo 2 | move=(0, 1) -> {'punks_left': 2, 'hippies_left': 3, 'punks_right': 1, 'hippies_right': 0, 'boat': 'left'}
Passo 3 | move=(2, 0) -> {'punks_left': 0, 'hippies_left': 3, 'punks_right': 3, 'hippies_right': 0, 'boat': 'right'}
Passo 4 | move=(1, 0) -> {'punks_left': 1, 'hippies_left': 3, 'punks_right': 2, 'hippies_right': 0, 'boat': 'left'}
Passo 5 | move=(0, 2) -> {'punks_left': 1, 'hippies_left': 1, 'punks_right': 2, 'hippies_right': 2, 'boat': 'right'}
Passo 6 | move=(1, 1) -> {'punks_left': 2, 'hippies_left': 2, 'punks_right': 1, 'hippies_right': 1, 'boat': 'left'}
Passo 7 | move=(0, 2) -> {'punks_left': 2, 'hippies_left': 0, 'punks_right': 1, 'hippies_right': 3, 'boat': 'right'}
Passo 8 | move=(1, 0) -> {'punks_left': 3, 'hippies_left':